<a href="https://colab.research.google.com/github/nurniahamid/Analisis_Sentimen-/blob/main/Analisis_Sentimen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install numpy==1.24.4 scipy==1.11.4 gensim==4.3.1

In [ ]:
!pip install Sastrawi

In [ ]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.5 MB/s eta 0:00:00


In [ ]:
!pip install textblob
!python -m textblob.download_corpora

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
Finished.


In [ ]:
import pandas as pd
import numpy as np

from sklearn.svm import SVC
from gensim.models import Word2Vec
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
review_capcut = pd.read_csv('/content/drive/MyDrive/Analisis Sentimen/review_capcut.csv')
review_capcut.head()

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,d90645c6-a681-431f-aef1-d7c7a88da4d5,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Capcut capucino centut,2,0,NaN,2025-04-27 7:10:48,NaN,NaN,NaN
1,3fe30fd2-597a-4272-bcdc-8c32c2eb74ff,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,aplikasi ini sangat cocok untuk mengedit,5,1,NaN,2025-04-27 1:52:50,NaN,NaN,NaN
2,1f79ded1-9241-432c-81d1-ca52141c2ce5,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Jujurly ini apk nya bagus cuman apk nya tuh ba...,5,0,NaN,2025-04-26 13:39:27,NaN,NaN,NaN
3,1ffa57cf-6315-4262-9f4b-7b9a2c00b347,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"PRA PRO PRA PRO MUAK GW DENGERIN, LU KENAPA SI...",1,1,14.2.0,2025-04-26 8:28:45,NaN,NaN,14.2.0
4,25fdbe8f-2dd2-42ed-a780-fd0ed4659bc7,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,apa apa Sekarang pro kadang iklan kadang ngebu...,1,4,14.0.0,2025-04-24 12:41:18,NaN,NaN,14.0.0


In [ ]:
review_capcut.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10997 entries, 0 to 10996
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              10997 non-null  object
 1   userName              10997 non-null  object
 2   userImage             10997 non-null  object
 3   content               10997 non-null  object
 4   score                 10997 non-null  int64 
 5   thumbsUpCount         10997 non-null  int64 
 6   reviewCreatedVersion  5767 non-null   object
 7   at                    10997 non-null  object
 8   replyContent          36 non-null     object
 9   repliedAt             36 non-null     object
 10  appVersion            5767 non-null   object
dtypes: int64(2), object(9)
memory usage: 945.2+ KB


In [ ]:
review_capcut= review_capcut[['userName','content', 'score', 'at']]

In [ ]:
review_capcut.isnull().sum()

,0
userName,0
content,0
score,0
at,0


In [ ]:
review_capcut.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10997 entries, 0 to 10996
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   userName  10997 non-null  object
 1   content   10997 non-null  object
 2   score     10997 non-null  int64 
 3   at        10997 non-null  object
dtypes: int64(1), object(3)
memory usage: 343.8+ KB


In [ ]:
review_capcut.duplicated().sum()

0

# **Text Preprocessing**

In [ ]:
#Tahapan Case Folding

def case_folding(content):
    if isinstance(content, str):
        content = content.lower()
    return content

review_capcut['case_folding'] = review_capcut['content'].apply(case_folding)
review_capcut.head()

,userName,content,score,at,case_folding
0,Pengguna Google,Capcut capucino centut,2,2025-04-27 7:10:48,capcut capucino centut
1,Pengguna Google,aplikasi ini sangat cocok untuk mengedit,5,2025-04-27 1:52:50,aplikasi ini sangat cocok untuk mengedit
2,Pengguna Google,Jujurly ini apk nya bagus cuman apk nya tuh ba...,5,2025-04-26 13:39:27,jujurly ini apk nya bagus cuman apk nya tuh ba...
3,Pengguna Google,"PRA PRO PRA PRO MUAK GW DENGERIN, LU KENAPA SI...",1,2025-04-26 8:28:45,"pra pro pra pro muak gw dengerin, lu kenapa si..."
4,Pengguna Google,apa apa Sekarang pro kadang iklan kadang ngebu...,1,2025-04-24 12:41:18,apa apa sekarang pro kadang iklan kadang ngebu...


In [ ]:
# Tahapan Cleansing
import re
def cleansing(content):
    if isinstance(content, str):
        content = re.sub(r'[?|$|.|!**2_:")(-+,]','', content)
        content = re.sub(r'@[A-Za-z0-9]+', '', content)
        content = re.sub(r'RT[\s]', '', content)
        content = re.sub(r"http\S+", '', content)
        content = re.sub(r'[0-9]+', '', content)
        content = re.sub(r'[^\w\s]', '', content)
        content = content.replace('\n', ' ')
        content = content.strip(' ')
    return content

review_capcut['cleansing'] = review_capcut['case_folding'].apply(cleansing)
review_capcut.head()

,userName,content,score,at,case_folding,cleansing
0,Pengguna Google,Capcut capucino centut,2,2025-04-27 7:10:48,capcut capucino centut,capcut capucino centut
1,Pengguna Google,aplikasi ini sangat cocok untuk mengedit,5,2025-04-27 1:52:50,aplikasi ini sangat cocok untuk mengedit,aplikasi ini sangat cocok untuk mengedit
2,Pengguna Google,Jujurly ini apk nya bagus cuman apk nya tuh ba...,5,2025-04-26 13:39:27,jujurly ini apk nya bagus cuman apk nya tuh ba...,jujurly ini apk nya bagus cuman apk nya tuh ba...
3,Pengguna Google,"PRA PRO PRA PRO MUAK GW DENGERIN, LU KENAPA SI...",1,2025-04-26 8:28:45,"pra pro pra pro muak gw dengerin, lu kenapa si...",pra pro pra pro muak gw dengerin lu kenapa sih...
4,Pengguna Google,apa apa Sekarang pro kadang iklan kadang ngebu...,1,2025-04-24 12:41:18,apa apa sekarang pro kadang iklan kadang ngebu...,apa apa sekarang pro kadang iklan kadang ngebu...


In [ ]:
#Tahapan tokenisasi

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt_tab')

review_capcut['tokenized'] = review_capcut['cleansing'].apply(word_tokenize)
review_capcut.head()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,userName,content,score,at,case_folding,cleansing,tokenized
0,Pengguna Google,Capcut capucino centut,2,2025-04-27 7:10:48,capcut capucino centut,capcut capucino centut,"[capcut, capucino, centut]"
1,Pengguna Google,aplikasi ini sangat cocok untuk mengedit,5,2025-04-27 1:52:50,aplikasi ini sangat cocok untuk mengedit,aplikasi ini sangat cocok untuk mengedit,"[aplikasi, ini, sangat, cocok, untuk, mengedit]"
2,Pengguna Google,Jujurly ini apk nya bagus cuman apk nya tuh ba...,5,2025-04-26 13:39:27,jujurly ini apk nya bagus cuman apk nya tuh ba...,jujurly ini apk nya bagus cuman apk nya tuh ba...,"[jujurly, ini, apk, nya, bagus, cuman, apk, ny..."
3,Pengguna Google,"PRA PRO PRA PRO MUAK GW DENGERIN, LU KENAPA SI...",1,2025-04-26 8:28:45,"pra pro pra pro muak gw dengerin, lu kenapa si...",pra pro pra pro muak gw dengerin lu kenapa sih...,"[pra, pro, pra, pro, muak, gw, dengerin, lu, k..."
4,Pengguna Google,apa apa Sekarang pro kadang iklan kadang ngebu...,1,2025-04-24 12:41:18,apa apa sekarang pro kadang iklan kadang ngebu...,apa apa sekarang pro kadang iklan kadang ngebu...,"[apa, apa, sekarang, pro, kadang, iklan, kadan..."


In [ ]:
#Tahapan Normalisasi
import json
kamus_file_path = '/content/drive/MyDrive/Analisis Sentimen/kamus_normalisasi.json'

with open(kamus_file_path, 'r', encoding='utf-8') as f:
    kamus_normalisasi = json.load(f)

def normalisasi_kalimat(token_list):
    return [kamus_normalisasi.get(token, token) for token in token_list]

review_capcut['normalisasi'] = review_capcut['tokenized'].apply(normalisasi_kalimat)
review_capcut.head()

,userName,content,score,at,case_folding,cleansing,tokenized,normalisasi
0,Pengguna Google,Capcut capucino centut,2,2025-04-27 7:10:48,capcut capucino centut,capcut capucino centut,"[capcut, capucino, centut]","[Capcut, capucino, centut]"
1,Pengguna Google,aplikasi ini sangat cocok untuk mengedit,5,2025-04-27 1:52:50,aplikasi ini sangat cocok untuk mengedit,aplikasi ini sangat cocok untuk mengedit,"[aplikasi, ini, sangat, cocok, untuk, mengedit]","[aplikasi, ini, sangat, cocok, untuk, mengedit]"
2,Pengguna Google,Jujurly ini apk nya bagus cuman apk nya tuh ba...,5,2025-04-26 13:39:27,jujurly ini apk nya bagus cuman apk nya tuh ba...,jujurly ini apk nya bagus cuman apk nya tuh ba...,"[jujurly, ini, apk, nya, bagus, cuman, apk, ny...","[jujurly, ini, aplikasi, nya, bagus, cuma, apl..."
3,Pengguna Google,"PRA PRO PRA PRO MUAK GW DENGERIN, LU KENAPA SI...",1,2025-04-26 8:28:45,"pra pro pra pro muak gw dengerin, lu kenapa si...",pra pro pra pro muak gw dengerin lu kenapa sih...,"[pra, pro, pra, pro, muak, gw, dengerin, lu, k...","[pra, premium, pra, premium, muak, saya, denge..."
4,Pengguna Google,apa apa Sekarang pro kadang iklan kadang ngebu...,1,2025-04-24 12:41:18,apa apa sekarang pro kadang iklan kadang ngebu...,apa apa sekarang pro kadang iklan kadang ngebu...,"[apa, apa, sekarang, pro, kadang, iklan, kadan...","[apa, apa, sekarang, premium, kadang, iklan, k..."


In [ ]:
#Tahapan Stemming
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stemming_kalimat(tokens):
    return [stemmer.stem(token) for token in tokens]

review_capcut['stemmed'] = review_capcut['normalisasi'].apply(stemming_kalimat)
review_capcut.head()


,userName,content,score,at,case_folding,cleansing,tokenized,normalisasi,stemmed
0,Pengguna Google,Capcut capucino centut,2,2025-04-27 7:10:48,capcut capucino centut,capcut capucino centut,"[capcut, capucino, centut]","[Capcut, capucino, centut]","[capcut, capucino, centut]"
1,Pengguna Google,aplikasi ini sangat cocok untuk mengedit,5,2025-04-27 1:52:50,aplikasi ini sangat cocok untuk mengedit,aplikasi ini sangat cocok untuk mengedit,"[aplikasi, ini, sangat, cocok, untuk, mengedit]","[aplikasi, ini, sangat, cocok, untuk, mengedit]","[aplikasi, ini, sangat, cocok, untuk, edit]"
2,Pengguna Google,Jujurly ini apk nya bagus cuman apk nya tuh ba...,5,2025-04-26 13:39:27,jujurly ini apk nya bagus cuman apk nya tuh ba...,jujurly ini apk nya bagus cuman apk nya tuh ba...,"[jujurly, ini, apk, nya, bagus, cuman, apk, ny...","[jujurly, ini, aplikasi, nya, bagus, cuma, apl...","[jujurly, ini, aplikasi, nya, bagus, cuma, apl..."
3,Pengguna Google,"PRA PRO PRA PRO MUAK GW DENGERIN, LU KENAPA SI...",1,2025-04-26 8:28:45,"pra pro pra pro muak gw dengerin, lu kenapa si...",pra pro pra pro muak gw dengerin lu kenapa sih...,"[pra, pro, pra, pro, muak, gw, dengerin, lu, k...","[pra, premium, pra, premium, muak, saya, denge...","[pra, premium, pra, premium, muak, saya, denge..."
4,Pengguna Google,apa apa Sekarang pro kadang iklan kadang ngebu...,1,2025-04-24 12:41:18,apa apa sekarang pro kadang iklan kadang ngebu...,apa apa sekarang pro kadang iklan kadang ngebu...,"[apa, apa, sekarang, pro, kadang, iklan, kadan...","[apa, apa, sekarang, premium, kadang, iklan, k...","[apa, apa, sekarang, premium, kadang, iklan, k..."


# **Pelabelan**

In [ ]:
from deep_translator import GoogleTranslator
from tqdm import tqdm

def join_tokens(tokens):
    return ' '.join(tokens)

review_capcut['normalisasi_text'] = review_capcut['normalisasi'].apply(join_tokens)

def translate_text(text):
    try:
        return GoogleTranslator(source='auto', target='en').translate(text)
    except Exception as e:
        return f"Error: {e}"

tqdm.pandas()
review_capcut['translated'] = review_capcut['normalisasi_text'].progress_apply(translate_text)

print(review_capcut[['normalisasi', 'translated']].head())

100%|██████████| 10997/10997 [1:20:06<00:00,  2.29it/s]

                                         normalisasi  \
0                         [Capcut, capucino, centut]   
1    [aplikasi, ini, sangat, cocok, untuk, mengedit]   
2  [jujurly, ini, aplikasi, nya, bagus, cuma, apl...   
3  [pra, premium, pra, premium, muak, saya, denge...   
4  [apa, apa, sekarang, premium, kadang, iklan, k...   

                                          translated  
0                             Capcut Capucino Centut  
1            This application is perfect for editing  
2  Honestly, this application is good, only the a...  
3  Pre Premium Premium I am fed up listening to y...  
4  What are the premiums now sometimes advertisem...  


In [ ]:
review_capcut = review_capcut.dropna(subset=['normalisasi_text'])
review_capcut = review_capcut[review_capcut['normalisasi_text'].str.strip() != '']
review_capcut.reset_index(drop=True)

,userName,content,score,at,case_folding,cleansing,tokenized,normalisasi,stemmed,normalisasi_text,translated
0,Pengguna Google,Capcut capucino centut,2,2025-04-27 7:10:48,capcut capucino centut,capcut capucino centut,"[capcut, capucino, centut]","[Capcut, capucino, centut]","[capcut, capucino, centut]",Capcut capucino centut,Capcut Capucino Centut
1,Pengguna Google,aplikasi ini sangat cocok untuk mengedit,5,2025-04-27 1:52:50,aplikasi ini sangat cocok untuk mengedit,aplikasi ini sangat cocok untuk mengedit,"[aplikasi, ini, sangat, cocok, untuk, mengedit]","[aplikasi, ini, sangat, cocok, untuk, mengedit]","[aplikasi, ini, sangat, cocok, untuk, edit]",aplikasi ini sangat cocok untuk mengedit,This application is perfect for editing
2,Pengguna Google,Jujurly ini apk nya bagus cuman apk nya tuh ba...,5,2025-04-26 13:39:27,jujurly ini apk nya bagus cuman apk nya tuh ba...,jujurly ini apk nya bagus cuman apk nya tuh ba...,"[jujurly, ini, apk, nya, bagus, cuman, apk, ny...","[jujurly, ini, aplikasi, nya, bagus, cuma, apl...","[jujurly, ini, aplikasi, nya, bagus, cuma, apl...",jujurly ini aplikasi nya bagus cuma aplikasi n...,"Honestly, this application is good, only the a..."
3,Pengguna Google,"PRA PRO PRA PRO MUAK GW DENGERIN, LU KENAPA SI...",1,2025-04-26 8:28:45,"pra pro pra pro muak gw dengerin, lu kenapa si...",pra pro pra pro muak gw dengerin lu kenapa sih...,"[pra, pro, pra, pro, muak, gw, dengerin, lu, k...","[pra, premium, pra, premium, muak, saya, denge...","[pra, premium, pra, premium, muak, saya, denge...",pra premium pra premium muak saya dengerin kam...,Pre Premium Premium I am fed up listening to y...
4,Pengguna Google,apa apa Sekarang pro kadang iklan kadang ngebu...,1,2025-04-24 12:41:18,apa apa sekarang pro kadang iklan kadang ngebu...,apa apa sekarang pro kadang iklan kadang ngebu...,"[apa, apa, sekarang, pro, kadang, iklan, kadan...","[apa, apa, sekarang, premium, kadang, iklan, k...","[apa, apa, sekarang, premium, kadang, iklan, k...",apa apa sekarang premium kadang iklan kadang b...,What are the premiums now sometimes advertisem...
...,...,...,...,...,...,...,...,...,...,...,...
10879,Angel Angelina,Capcut ini sangat lah membuat ku bahagia dan k...,5,2025-03-13 8:31:41,capcut ini sangat lah membuat ku bahagia dan k...,capcut ini sangat lah membuat ku bahagia dan k...,"[capcut, ini, sangat, lah, membuat, ku, bahagi...","[Capcut, ini, sangat, lah, membuat, ku, bahagi...","[capcut, ini, sangat, lah, buat, ku, bahagia, ...",Capcut ini sangat lah membuat ku bahagia dan k...,This capcut really makes me happy and sometime...
10880,Romeesa Bivania,Heh owner nya cap cut semua fitur nya tolong d...,2,2025-03-13 8:30:58,heh owner nya cap cut semua fitur nya tolong d...,heh owner nya cap cut semua fitur nya tolong d...,"[heh, owner, nya, cap, cut, semua, fitur, nya,...","[heh, owner, nya, cap, cut, semua, fitur, nya,...","[heh, owner, nya, cap, cut, semua, fitur, nya,...",heh owner nya cap cut semua fitur nya tolong d...,"heh the owner stamp cut all the features, plea..."
10881,Khumaira Inara,Aplikasi e kok sekarang jelek banget... Banyak...,1,2025-03-13 8:25:22,aplikasi e kok sekarang jelek banget... banyak...,aplikasi e kok sekarang jelek banget banyak ik...,"[aplikasi, e, kok, sekarang, jelek, banget, ba...","[aplikasi, e, kok, sekarang, buruk, banget, ba...","[aplikasi, e, kok, sekarang, buruk, banget, ba...",aplikasi e kok sekarang buruk banget banyak ik...,The E application is now so bad now a lot of a...
10882,laa 4aa,terlalu banyak iklan dan kadang kalo saya masu...,4,2025-03-13 8:20:50,terlalu banyak iklan dan kadang kalo saya masu...,terlalu banyak iklan dan kadang kalo saya masu...,"[terlalu, banyak, iklan, dan, kadang, kalo, sa...","[terlalu, banyak, iklan, dan, kadang, kalau, s...","[terlalu, banyak, iklan, dan, kadang, kalau, s...",terlalu banyak iklan dan kadang kalau saya mas...,too many advertisements and sometimes when I e...


In [ ]:
def get_polarity(text):
    try:
        text = str(text)
        return TextBlob(text).sentiment.polarity
    except:
        return None

def get_subjectivity(text):
    try:
        text = str(text)
        return TextBlob(text).sentiment.subjectivity
    except:
        return None

def get_sentiment_label(text):
    try:
        text = str(text)
        polarity = TextBlob(text).sentiment.polarity
        if polarity > 0:
            return 'positif'
        elif polarity < 0:
            return 'negatif'
        else:
            return 'netral'
    except:
        return 'error'

review_capcut['sentiment'] = review_capcut['translated'].progress_apply(get_sentiment_label)
review_capcut['polarity'] = review_capcut['translated'].progress_apply(get_polarity)
review_capcut['subjektivitas'] = review_capcut['translated'].progress_apply(get_subjectivity)
review_capcut.head()

100%|██████████| 10884/10884 [00:03<00:00, 3615.70it/s]


,userName,content,score,at,case_folding,cleansing,tokenized,normalisasi,stemmed,normalisasi_text,translated,sentiment,polarity,subjektivitas
0,Pengguna Google,Capcut capucino centut,2,2025-04-27 7:10:48,capcut capucino centut,capcut capucino centut,"[capcut, capucino, centut]","[Capcut, capucino, centut]","[capcut, capucino, centut]",Capcut capucino centut,Capcut Capucino Centut,netral,0.000000,0.000000
1,Pengguna Google,aplikasi ini sangat cocok untuk mengedit,5,2025-04-27 1:52:50,aplikasi ini sangat cocok untuk mengedit,aplikasi ini sangat cocok untuk mengedit,"[aplikasi, ini, sangat, cocok, untuk, mengedit]","[aplikasi, ini, sangat, cocok, untuk, mengedit]","[aplikasi, ini, sangat, cocok, untuk, edit]",aplikasi ini sangat cocok untuk mengedit,This application is perfect for editing,positif,1.000000,1.000000
2,Pengguna Google,Jujurly ini apk nya bagus cuman apk nya tuh ba...,5,2025-04-26 13:39:27,jujurly ini apk nya bagus cuman apk nya tuh ba...,jujurly ini apk nya bagus cuman apk nya tuh ba...,"[jujurly, ini, apk, nya, bagus, cuman, apk, ny...","[jujurly, ini, aplikasi, nya, bagus, cuma, apl...","[jujurly, ini, aplikasi, nya, bagus, cuma, apl...",jujurly ini aplikasi nya bagus cuma aplikasi n...,"Honestly, this application is good, only the a...",positif,0.433333,0.833333
3,Pengguna Google,"PRA PRO PRA PRO MUAK GW DENGERIN, LU KENAPA SI...",1,2025-04-26 8:28:45,"pra pro pra pro muak gw dengerin, lu kenapa si...",pra pro pra pro muak gw dengerin lu kenapa sih...,"[pra, pro, pra, pro, muak, gw, dengerin, lu, k...","[pra, premium, pra, premium, muak, saya, denge...","[pra, premium, pra, premium, muak, saya, denge...",pra premium pra premium muak saya dengerin kam...,Pre Premium Premium I am fed up listening to y...,negatif,-0.137500,0.526667
4,Pengguna Google,apa apa Sekarang pro kadang iklan kadang ngebu...,1,2025-04-24 12:41:18,apa apa sekarang pro kadang iklan kadang ngebu...,apa apa sekarang pro kadang iklan kadang ngebu...,"[apa, apa, sekarang, pro, kadang, iklan, kadan...","[apa, apa, sekarang, premium, kadang, iklan, k...","[apa, apa, sekarang, premium, kadang, iklan, k...",apa apa sekarang premium kadang iklan kadang b...,What are the premiums now sometimes advertisem...,netral,0.000000,0.000000


In [ ]:
review_capcut['sentiment'].value_counts()

,count
sentiment,
positif,6982
netral,2689
negatif,1213


# **TF-IDF**

In [ ]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vectorizer.fit_transform(review_capcut['normalisasi_text']).toarray()

print(X_tfidf.shape)

(10884, 5000)


# **Model**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, review_capcut['sentiment'], test_size=0.2, random_state=42)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Skema 1 (baru): Random Forest + TF-IDF + Pembagian Data 70/30")
print("Akurasi:", accuracy_score(y_test, y_pred))
print("Laporan Klasifikasi:\n", classification_report(y_test, y_pred))

Training set: (8707, 5000), Test set: (2177, 5000)
Skema 1 (baru): Random Forest + TF-IDF + Pembagian Data 70/30
Akurasi: 0.825447864033073
Laporan Klasifikasi:
               precision    recall  f1-score   support

     negatif       0.78      0.26      0.39       252
      netral       0.80      0.77      0.78       538
     positif       0.84      0.95      0.89      1387

    accuracy                           0.83      2177
   macro avg       0.81      0.66      0.69      2177
weighted avg       0.82      0.83      0.81      2177



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, review_capcut['sentiment'], test_size=0.2, random_state=42)

svm_model = SVC(kernel='linear')
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

print("Skema 2: SVM + TF-IDF + Pembagian Data 80/20")
print("Akurasi:", accuracy_score(y_test, y_pred_svm))
print("Laporan Klasifikasi:\n", classification_report(y_test, y_pred_svm))


Skema 2: SVM + TF-IDF + Pembagian Data 80/20
Akurasi: 0.8327974276527331
Laporan Klasifikasi:
               precision    recall  f1-score   support

     negatif       0.71      0.50      0.59       252
      netral       0.75      0.81      0.78       538
     positif       0.88      0.90      0.89      1387

    accuracy                           0.83      2177
   macro avg       0.78      0.74      0.75      2177
weighted avg       0.83      0.83      0.83      2177



In [ ]:
sentences = review_capcut['normalisasi_text'].apply(lambda x: x.split())
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

def get_word2vec_features(sentence):
    words = sentence.split()
    word_vectors = [w2v_model.wv[word] for word in words if word in w2v_model.wv]
    if len(word_vectors) == 0:
        return np.zeros(100)
    return np.mean(word_vectors, axis=0)

X_word2vec = np.array([get_word2vec_features(sentence) for sentence in review_capcut['normalisasi_text']])
X_train, X_test, y_train, y_test = train_test_split(X_word2vec, review_capcut['sentiment'], test_size=0.2, random_state=42)

svm_model_w2v = SVC(kernel='linear')
svm_model_w2v.fit(X_train, y_train)

y_pred_svm_w2v = svm_model_w2v.predict(X_test)

print("Skema 3: SVM + Word2Vec + Pembagian Data 80/20")
print("Akurasi:", accuracy_score(y_test, y_pred_svm_w2v))
print("Laporan Klasifikasi:\n", classification_report(y_test, y_pred_svm_w2v))


Skema 3: SVM + Word2Vec + Pembagian Data 80/20
Akurasi: 0.7175011483693156
Laporan Klasifikasi:
               precision    recall  f1-score   support

     negatif       0.80      0.02      0.03       252
      netral       0.62      0.56      0.58       538
     positif       0.75      0.91      0.82      1387

    accuracy                           0.72      2177
   macro avg       0.72      0.49      0.48      2177
weighted avg       0.72      0.72      0.67      2177

